# Baseline — Sınıf Dengesizliği (SMOTE & Undersampling)

SMS Spam veri seti 87:13 oranında dengesizdir. Bu notebook SMOTE ve Random Undersampling yöntemlerini
baseline ile karşılaştırarak resampling'in model performansına etkisini ölçer.

### Kütüphaneler

In [7]:
import pandas as pd
import numpy as np
import time
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb


### Veri Yükleme ve Bölme

In [8]:
df = pd.read_csv('spam_cleaned.csv', encoding='latin-1')

X, y = df['sms'], df['type']
print(f"Veri seti: {len(df)} satır | ham: {(y==0).sum()}, spam: {(y==1).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111, stratify=y
)
print(f"Eğitim: {len(X_train)} | Test: {len(X_test)} | Train spam oranı: {y_train.mean()*100:.1f}%")


Veri seti: 5570 satır | ham: 4823, spam: 747
Eğitim: 3899 | Test: 1671 | Train spam oranı: 13.4%


### Vektörleştirme ve Resampling

SMOTE ve Random Undersampling **yalnızca eğitim setine** uygulanır. Test seti orijinal dağılımı korur (data leakage önlemi).

In [9]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# SMOTE
smote = SMOTE(random_state=42)
X_tr_smote, y_tr_smote = smote.fit_resample(X_train_vec, y_train)
print(f"Orijinal  — ham: {(y_train==0).sum()}, spam: {(y_train==1).sum()}")
print(f"SMOTE     — ham: {(y_tr_smote==0).sum()}, spam: {(y_tr_smote==1).sum()}")

# Undersampling
rus = RandomUnderSampler(random_state=42)
X_tr_us, y_tr_us = rus.fit_resample(X_train_vec, y_train)
print(f"Undersamp — ham: {(y_tr_us==0).sum()}, spam: {(y_tr_us==1).sum()}")


Orijinal  — ham: 3376, spam: 523
SMOTE     — ham: 3376, spam: 3376
Undersamp — ham: 523, spam: 523


### Değerlendirme Fonksiyonu

In [10]:
import scipy.stats as stats

def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    try:
        y_score = clf.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_score = clf.decision_function(X_te)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='accuracy')
    cv_acc = cv_scores.mean()
    cv_std = cv_scores.std()
    ci = stats.t.interval(0.95, df=len(cv_scores)-1, loc=cv_acc, scale=stats.sem(cv_scores))
    return {
        'Model':        name,
        'Accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'Precision':    round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_te, y_pred), 4),
        'F1':           round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_te, y_score), 4),
        'CV-Acc':       round(cv_acc, 4),
        'CV-Std':       round(cv_std, 4),
        'CI-95% Low':   round(ci[0], 4),
        'CI-95% High':  round(ci[1], 4),
        'Time(s)':      round(elapsed, 4),
    }

### Karşılaştırma: Baseline vs SMOTE vs Undersampling

Her yöntem için tüm modeller çalıştırılır. Sonuçlar ve delta tablosu yan yana raporlanır.

In [11]:
classifiers = [
    ('LR',  LogisticRegression(max_iter=1000)),
    ('DT',  DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('SVM', SVC(kernel='linear', probability=True)),
    ('NC',  NearestCentroid()),
]

datasets = {
    'Baseline':      (X_train_vec, y_train),
    'SMOTE':         (X_tr_smote,  y_tr_smote),
    'Undersampling': (X_tr_us,     y_tr_us),
}

all_rows = []
for ds_name, (X_tr, y_tr) in datasets.items():
    for clf_name, clf in classifiers:
        row = evaluate(clf_name, copy.deepcopy(clf), X_tr, y_tr, X_test_vec, y_test)
        row['Dataset'] = ds_name
        all_rows.append(row)

cols = ['Dataset', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC',
        'CV-Acc', 'CV-Std', 'CI-95% Low', 'CI-95% High', 'Time(s)']
df_results = pd.DataFrame(all_rows)[cols]
print(df_results.to_string(index=False))

# Delta tablosu
print("\n=== Delta: yöntem − Baseline ===")
base = df_results[df_results['Dataset']=='Baseline'].set_index('Model')
for ds in ['SMOTE', 'Undersampling']:
    print(f"\n--- {ds} ---")
    cur = df_results[df_results['Dataset']==ds].set_index('Model')
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC',
               'CV-Acc', 'CV-Std', 'CI-95% Low', 'CI-95% High']
    delta = (cur[metrics] - base[metrics]).round(4)
    print(delta.to_string())


      Dataset Model  Accuracy  Precision  Recall     F1  ROC-AUC  CV-Acc  CV-Std  CI-95% Low  CI-95% High  Time(s)
     Baseline    LR    0.9838     0.9713  0.9062 0.9376   0.9877  0.9820  0.0044      0.9760       0.9881   0.0135
     Baseline    DT    0.9731     0.9124  0.8839 0.8980   0.9354  0.9690  0.0021      0.9661       0.9718   0.0353
     Baseline   KNN    0.9659     0.9326  0.8036 0.8633   0.9316  0.9623  0.0029      0.9583       0.9663   0.0003
     Baseline   SVM    0.9844     0.9670  0.9152 0.9404   0.9831  0.9820  0.0039      0.9766       0.9875   0.7509
     Baseline    NC    0.9575     0.8964  0.7723 0.8297   0.9955  0.9487  0.0049      0.9419       0.9555   0.0740
        SMOTE    LR    0.9767     0.9004  0.9286 0.9143   0.9710  0.9753  0.0026      0.9717       0.9789   0.0231
        SMOTE    DT    0.8654     0.4987  0.8482 0.6281   0.8640  0.9224  0.0065      0.9133       0.9315   0.0756
        SMOTE   KNN    0.5356     0.2223  0.9866 0.3629   0.8262  0.7121  0.0159